# CEFR-Gated PMT — Training Resource Profiling

This notebook performs a dedicated hardware-profiling run for the proposed
**CEFR-Gated PrefixMemory-Tuning (PMT)** architecture.

The other thesis methods record their resource measurements directly inside
their respective training notebooks. A separate profiling run is used here
only for the proposed CEFR-Gated PMT controller.

The purpose of this notebook is to measure:

- total training and validation time;
- peak allocated GPU memory;
- average GPU utilization;
- hardware configuration;
- trainable and frozen parameter counts.

This is a **resource-profiling rerun**, not the canonical model-training
experiment used for the thesis accuracy results. No model checkpoint is
published or retained by this notebook.

The profiling run uses the same CEFR-Gated PMT architecture:

- frozen `meta-llama/Llama-3.1-8B-Instruct` backbone;
- 32 full `4096 × 4096` memory matrices;
- six trainable 4096-dimensional CEFR embeddings;
- one learnable scaling scalar `alpha`.

Total trainable controller parameters:

**536,895,489**

In [ ]:
# ============================================================
# 0. ENVIRONMENT SETUP
# ============================================================
!pip install -q transformers torch pandas scikit-learn tqdm nvidia-ml-py

import os
import gc
import time
import threading
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import pynvml
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# ============================================================
# 1. GOOGLE DRIVE & PATHS
# ============================================================
drive.mount("/content/drive")

BALANCED_CSV_PATH = "/content/drive/MyDrive/Your_Path/"
    "balanced_cefr_steering_subset.csv"
PROFILE_DIR = "/content/drive/MyDrive/Your_Path/"
    "cefr_gated_pmt_resource_profiling"
os.makedirs(PROFILE_DIR, exist_ok=True)
PROFILE_TXT_PATH = os.path.join(PROFILE_DIR, "cefr_gated_pmt_hardware_profile.txt")

# ============================================================
# 2. TRAINING CONFIGURATION
# ============================================================
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
TRAIN_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
VAL_BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 2e-4
MAX_LENGTH = 256
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ============================================================
# 3. DEVICE VERIFICATION
# ============================================================
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this profiling run.")

device = "cuda"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_TOTAL_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print("=" * 70 + "\nHARDWARE\n" + "=" * 70)
print(f"GPU               : {GPU_NAME}")
print(f"Total GPU Memory  : {GPU_TOTAL_MEMORY_GB:.2f} GB")
print(f"PyTorch Version   : {torch.__version__}")
print("=" * 70)

# ============================================================
# 4. GPU UTILIZATION MONITOR
# ============================================================
class GPUUtilizationMonitor:
    def __init__(self, gpu_index=0, polling_interval=0.5):
        self.gpu_index, self.polling_interval = gpu_index, polling_interval
        self.utilization_samples = []
        self._stop_event = threading.Event()
        self._thread = None
        pynvml.nvmlInit()
        self.handle = pynvml.nvmlDeviceGetHandleByIndex(gpu_index)

    def _monitor(self):
        while not self._stop_event.is_set():
            try:
                utilization = pynvml.nvmlDeviceGetUtilizationRates(self.handle)
                self.utilization_samples.append(float(utilization.gpu))
            except Exception:
                pass
            time.sleep(self.polling_interval)

    def start(self):
        self.utilization_samples = []
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._monitor, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join()
        try:
            pynvml.nvmlShutdown()
        except Exception:
            pass

    def get_average_utilization(self):
        return float(np.mean(self.utilization_samples)) if self.utilization_samples else 0.0

# ============================================================
# 5. LOAD BALANCED 6K DATASET
# ============================================================
print(f"\nLoading balanced steering dataset:\n{BALANCED_CSV_PATH}")
balanced_df = pd.read_csv(BALANCED_CSV_PATH).dropna(subset=["cefr", "clean_text", "topic_title"])
balanced_df["cefr"] = balanced_df["cefr"].astype(str).str.upper().str.strip()

label_map = {"A1": 0, "A2": 1, "B1": 2, "B2": 3, "C1": 4, "C2": 5}
print(f"\nDataset size: {len(balanced_df):,}\n\nClass distribution:\n{balanced_df['cefr'].value_counts().sort_index()}")

# ============================================================
# 6. SAME 90/10 STRATIFIED SPLIT
# ============================================================
train_df, val_df = train_test_split(
    balanced_df, test_size=0.10, stratify=balanced_df["cefr"], random_state=RANDOM_SEED
)
print(f"\nTraining samples   : {len(train_df):,}\nValidation samples : {len(val_df):,}")

# ============================================================
# 7. LOAD TOKENIZER & FROZEN LLAMA BACKBONE
# ============================================================
print("\nLoading frozen Llama-3.1-8B-Instruct backbone...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map={"": torch.cuda.current_device()})
for param in base_model.parameters():
    param.requires_grad = False
base_model.eval()

# ============================================================
# 8. FINAL PROPOSED PURE CEFR-GATED PMT
# ============================================================
class PureCEFRPMTController(nn.Module):
    def __init__(self, num_layers=32, hidden_dim=4096, num_classes=6):
        super().__init__()
        self.num_layers, self.hidden_dim = num_layers, hidden_dim
        self.alpha = nn.Parameter(torch.tensor(0.1))
        self.cefr_embeddings = nn.Embedding(num_classes, hidden_dim)
        self.M = nn.Parameter(torch.randn(num_layers, hidden_dim, hidden_dim) * (hidden_dim ** -0.5))

    def forward(self, cefr_ids):
        return self.cefr_embeddings(cefr_ids)

prefix_memory_controller = PureCEFRPMTController().to(device=device, dtype=torch.bfloat16)

# ============================================================
# 9. PARAMETER COUNTS
# ============================================================
controller_trainable_params = sum(p.numel() for p in prefix_memory_controller.parameters() if p.requires_grad)
backbone_total_params = sum(p.numel() for p in base_model.parameters())
backbone_trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)

print("\n" + "=" * 70 + "\nPARAMETER VERIFICATION\n" + "=" * 70)
print(f"Frozen backbone params     : {backbone_total_params:,}")
print(f"Trainable backbone params  : {backbone_trainable_params:,}")
print(f"PMT trainable params       : {controller_trainable_params:,}")

assert controller_trainable_params == 536_895_489, f"Unexpected CEFR-gated PMT parameter count. Observed {controller_trainable_params:,}."
assert backbone_trainable_params == 0, "Backbone is not completely frozen."
print("=" * 70)

# ============================================================
# 10. PURE PMT FORWARD HOOKS
# ============================================================
current_batch_cefr_vector = None

def make_pure_pmt_hook(layer_idx):
    def hook_fn(module, args, kwargs, output):
        global current_batch_cefr_vector
        if current_batch_cefr_vector is None: return output

        hidden_states = args[0] if len(args) > 0 else kwargs.get("hidden_states")
        if hidden_states is None: return output

        attn_output = output[0]
        if current_batch_cefr_vector.shape[0] != attn_output.shape[0]: return output

        phi_X = F.elu(hidden_states)
        gated_X = phi_X * current_batch_cefr_vector.unsqueeze(1)
        prefix_memory_bias = torch.matmul(gated_X, prefix_memory_controller.M[layer_idx])
        final_attn_output = attn_output + (prefix_memory_controller.alpha * prefix_memory_bias)

        return (final_attn_output,) + output[1:]
    return hook_fn

hook_handles = []
for layer_idx in range(32):
    handle = base_model.model.layers[layer_idx].self_attn.register_forward_hook(make_pure_pmt_hook(layer_idx), with_kwargs=True)
    hook_handles.append(handle)

# ============================================================
# 11. TOPIC-ALIGNED DATASET
# ============================================================
class CEFRTopicDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        cefr_str = str(row["cefr"]).strip().upper()
        topic_title = str(row["topic_title"]).strip()

        prompt = (
            f"You are an expert English language teacher demonstrating CEFR proficiency levels. "
            f"Your task is to write a flawless, grammatically correct text responding to this prompt: '{topic_title}'. "
            f"If the requested target level is A1/A2, use very simple vocabulary, short sentences, and primitive structures. "
            f"If the requested target level is C1/C2, utilize highly advanced vocabulary, idioms, and complex sentence patterns. "
            f"Write only the direct response. Do not write any meta-commentary, greetings, or conversational pleasantries."
        )

        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        target_text = str(row["clean_text"]).strip()
        full_text = formatted_prompt + target_text + self.tokenizer.eos_token

        prompt_enc = self.tokenizer(formatted_prompt, add_special_tokens=False)
        full_enc = self.tokenizer(full_text, truncation=True, max_length=self.max_length, padding="max_length")

        labels = list(full_enc["input_ids"])
        prompt_len = len(prompt_enc["input_ids"])

        for i in range(min(prompt_len, self.max_length)): labels[i] = -100
        for i, attention_value in enumerate(full_enc["attention_mask"]):
            if attention_value == 0: labels[i] = -100

        return {
            "input_ids": torch.tensor(full_enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(full_enc["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "cefr_id": torch.tensor(label_map[cefr_str], dtype=torch.long)
        }

# ============================================================
# 12. DATA LOADERS
# ============================================================
train_dataset = CEFRTopicDataset(train_df, tokenizer, max_length=MAX_LENGTH)
val_dataset = CEFRTopicDataset(val_df, tokenizer, max_length=MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

effective_batch_size = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS

print("\n" + "=" * 70 + "\nTRAINING CONFIGURATION\n" + "=" * 70)
print(f"Train batch size        : {TRAIN_BATCH_SIZE}")
print(f"Gradient accumulation   : {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size    : {effective_batch_size}")
print(f"Validation batch size   : {VAL_BATCH_SIZE}")
print(f"Epochs                  : {EPOCHS}")
print(f"Max sequence length     : {MAX_LENGTH}")
print("=" * 70)

# ============================================================
# 13. OPTIMIZER & 14. PRE-PROFILING CLEANUP
# ============================================================
optimizer = torch.optim.AdamW(prefix_memory_controller.parameters(), lr=LEARNING_RATE)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

# ============================================================
# 15. START HARDWARE PROFILER & 16. TRAINING + VALIDATION
# ============================================================
gpu_monitor = GPUUtilizationMonitor(gpu_index=0, polling_interval=0.5)
gpu_monitor.start()
torch.cuda.synchronize()
training_start_time = time.perf_counter()

print("\n🚀 Starting CEFR-gated PMT hardware profiling run...")

for epoch in range(EPOCHS):
    prefix_memory_controller.train()
    optimizer.zero_grad(set_to_none=True)
    total_train_loss, num_train_batches = 0.0, 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS} [Train]", leave=False)

    for batch_idx, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        cefr_ids = batch["cefr_id"].to(device, non_blocking=True)

        current_batch_cefr_vector = prefix_memory_controller(cefr_ids)
        outputs = base_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        raw_loss = outputs.loss
        loss = raw_loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward()

        should_step = ((batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0) or (batch_idx + 1 == len(train_loader))
        if should_step:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        total_train_loss += raw_loss.item()
        num_train_batches += 1
        progress_bar.set_postfix({"Loss": f"{raw_loss.item():.4f}"})

    avg_train_loss = total_train_loss / max(num_train_batches, 1)

    prefix_memory_controller.eval()
    total_val_loss, num_val_batches = 0.0, 0

    with torch.no_grad():
        val_progress_bar = tqdm(val_loader, desc=f"Epoch {epoch + 1}/{EPOCHS} [Validation]", leave=False)
        for batch in val_progress_bar:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            cefr_ids = batch["cefr_id"].to(device, non_blocking=True)

            current_batch_cefr_vector = prefix_memory_controller(cefr_ids)
            outputs = base_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

            total_val_loss += outputs.loss.item()
            num_val_batches += 1

    avg_val_loss = total_val_loss / max(num_val_batches, 1)
    print(f"Epoch {epoch + 1}/{EPOCHS} complete | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

# ============================================================
# 17. STOP PROFILING
# ============================================================
torch.cuda.synchronize()
training_end_time = time.perf_counter()
gpu_monitor.stop()

total_time_seconds = training_end_time - training_start_time
total_time_hours = total_time_seconds / 3600
peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
avg_gpu_utilization = gpu_monitor.get_average_utilization()

# ============================================================
# 18. HARDWARE PROFILING REPORT & 19. SAVE
# ============================================================
profile_report = f"""============================================================
CEFR-Gated PMT Hardware Profiling Report
============================================================
Model: Proposed Pure CEFR-Gated PrefixMemory-Tuning
Backbone: {MODEL_ID}

Hardware:
    GPU: {GPU_NAME}
    GPU total memory: {GPU_TOTAL_MEMORY_GB:.2f} GB

Dataset:
    Total samples: {len(balanced_df):,}
    Training samples: {len(train_df):,}
    Validation samples: {len(val_df):,}

Training Configuration:
    Epochs: {EPOCHS}
    Maximum sequence length: {MAX_LENGTH}
    Train batch size: {TRAIN_BATCH_SIZE}
    Gradient accumulation steps: {GRADIENT_ACCUMULATION_STEPS}
    Effective train batch size: {effective_batch_size}
    Validation batch size: {VAL_BATCH_SIZE}
    Learning rate: {LEARNING_RATE}

Parameterization:
    Frozen backbone parameters: {backbone_total_params:,}
    Trainable backbone parameters: {backbone_trainable_params:,}
    CEFR-gated PMT trainable parameters: {controller_trainable_params:,}

Hardware Profiling:
    Total training + validation time:
        {total_time_seconds:.2f} seconds ({total_time_hours:.2f} hours)
    Peak GPU memory allocated: {peak_vram_gb:.2f} GB
    Average GPU utilization: {avg_gpu_utilization:.1f} %
============================================================"""

with open(PROFILE_TXT_PATH, "w", encoding="utf-8") as f:
    f.write(profile_report)

print("\n" + profile_report)
print(f"\n✔ Hardware profile saved to:\n{PROFILE_TXT_PATH}")

# ============================================================
# 20. CLEANUP
# ============================================================
current_batch_cefr_vector = None
for handle in hook_handles: handle.remove()

del optimizer, prefix_memory_controller, base_model
gc.collect()
torch.cuda.empty_cache()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HARDWARE
GPU               : NVIDIA A100-SXM4-80GB
Total GPU Memory  : 79.25 GB
PyTorch Version   : 2.11.0+cu128

Loading balanced steering dataset:
/content/drive/MyDrive/Mohammd_Thesis/subsets/efcamdat_balanced_subset_training.csv

Dataset size: 5,568

Class distribution:
cefr
A1    928
A2    928
B1    928
B2    928
C1    928
C2    928
Name: count, dtype: int64

Training samples   : 5,011
Validation samples : 557

Loading frozen Llama-3.1-8B-Instruct backbone...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Epoch 2/3 [Validation]:   0%|          | 0/18 [00:00<?, ?it/s]

Epoch 2/3 complete | Train Loss: 2.1187 | Val Loss: 2.5027


Epoch 3/3 [Train]:   0%|          | 0/314 [00:00<?, ?it/s]

Epoch 3/3 [Validation]:   0%|          | 0/18 [00:00<?, ?it/s]

Epoch 3/3 complete | Train Loss: 1.4196 | Val Loss: 2.7717

CEFR-Gated PMT Hardware Profiling Report
Model: Proposed Pure CEFR-Gated PrefixMemory-Tuning
Backbone: meta-llama/Llama-3.1-8B-Instruct

Hardware:
    GPU: NVIDIA A100-SXM4-80GB
    GPU total memory: 79.25 GB

Dataset:
    Total samples: 5,568
    Training samples: 5,011
    Validation samples: 557

Training Configuration:
    Epochs: 3
    Maximum sequence length: 256
    Train batch size: 16
    Gradient accumulation steps: 2
    Effective train batch size: 32
    Validation batch size: 32
    Learning rate: 0.0002

Parameterization:
    Frozen backbone parameters: 8,030,261,248
    Trainable backbone parameters: 0
    CEFR-gated PMT trainable parameters: 536,895,489

Hardware Profiling:
    Total training + validation time:
        875.83 seconds (0.24 hours)
    Peak GPU memory allocated: 48.74 GB
    Average GPU utilization: 96.4 %

✔ Hardware profile saved to:
/content/drive/MyDrive/Mohammd_Thesis/Training_Logs/Pure_PMT_

## Profiling Summary

The dedicated CEFR-Gated PMT profiling run produced the following resource
measurements:

| Metric | Result |
|---|---:|
| GPU | NVIDIA A100-SXM4-80GB |
| Trainable controller parameters | 536,895,489 |
| Frozen backbone parameters | 8,030,261,248 |
| Training + validation time | 875.83 s |
| Training + validation time | 0.24 h |
| Peak allocated GPU memory | 48.74 GB |
| Average GPU utilization | 96.4% |

These values characterize the resource requirements of the proposed
CEFR-Gated PMT architecture under the profiling configuration used in this
notebook.

The profiling-run losses are not used as the model-quality results reported
for the main CEFR-Gated PMT experiment.